## Dependencies

In [1]:
import ir_datasets
import nltk
import re
import math
import pandas as pd
import numpy as np
import pymongo
import joblib
import numpy as np
from pymongo import MongoClient
from gensim.models import Word2Vec
import pandas as pd
import numpy as np
import joblib
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from gensim.models import Word2Vec
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from itertools import islice

import itertools
from gensim.models import Word2Vec
import logging

/Users/fadi/Documents/active/information-retrival/103/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/fadi/Documents/active/information-retrival/103/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class DataBase:
    def __init__(self):
        self.client = MongoClient("mongodb://localhost:27017/")
        self.db = self.client["lottesearch"]
        
        self.dataset = ir_datasets.load("lotte/lifestyle/dev/search")
        self.qrels = self.dataset.qrels_iter()
        self.queries = self.dataset.queries_iter()
        self.documents = self.db["documents"]
        self.queries = self.db["queries"]
        self.qrels = self.db["qrels"]
        self.documents_processed = self.db["documents_processed"]
        self.queries_processed = self.db["queries_processed"]
    
    def store_documents(self):
        self.documents.delete_many({})
        self.queries.delete_many({})
        self.qrels.delete_many({})

        docs_batch = []

        for doc in self.dataset.docs_iter():
            docs_batch.append({
                "_id": doc.doc_id,
                "text": doc.text
            })

            if len(docs_batch) >= 1000:
                self.documents.insert_many(docs_batch)
                docs_batch = []

        if docs_batch:
            self.documents.insert_many(docs_batch)

        print("Documents stored")

        queries_batch = []

        for query in self.dataset.queries_iter():
            queries_batch.append({
                "_id": query.query_id,
                "text": query.text
            })

            if len(queries_batch) >= 1000:
                self.queries.insert_many(queries_batch)
                queries_batch = []

        if queries_batch:
            self.queries.insert_many(queries_batch)

        print("Queries stored")

        qrels_batch = []

        for qrel in self.dataset.qrels_iter():
            qrels_batch.append({
                "query_id": qrel.query_id,
                "doc_id": qrel.doc_id,
                "relevance": qrel.relevance,
                "iteration": qrel.iteration
            })

            if len(qrels_batch) >= 1000:
                self.qrels.insert_many(qrels_batch)
                qrels_batch = []

        if qrels_batch:
            self.qrels.insert_many(qrels_batch)

        print("Qrels stored")
        print("Done")

    def summary(self):
        print("Documents:", self.documents.count_documents({}))
        print("Queries:", self.queries.count_documents({}))
        print("Qrels:", self.qrels.count_documents({}))
        print("Documents Processed:", self.documents_processed.count_documents({}))
        print("Queries Processed:", self.queries_processed.count_documents({}))


## Pre-processing

In [4]:
class Preprocessor:
    def __init__(self, remove_stopwords=True):
        nltk.download('stopwords')
        self.remove_stopwords = remove_stopwords
        self.stop_words = set(stopwords.words("english"))
        self.lemmatizer = WordNetLemmatizer()
        self.database = DataBase()

    def normalize(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r"[^a-z0-9\s]", " ", text)  
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def tokenize(self, text: str):
        return text.split()

    def remove_stopwords_fn(self, tokens):
        if not self.remove_stopwords:
            return tokens
        return [t for t in tokens if t not in self.stop_words]

    def lemmatize(self, tokens):
        return [self.lemmatizer.lemmatize(t) for t in tokens]

    def process(self, text: str):
        text = self.normalize(text)
        tokens = self.tokenize(text)
        tokens = self.remove_stopwords_fn(tokens)
        tokens = self.lemmatize(tokens)
        return tokens
    
    def process_documents(self, documents, docs_proc):
        docs_proc.delete_many({}) 
        batch = []
        for doc in documents.find({}):
            batch.append({
                "_id": doc["_id"],
                "tokens": self.process(doc["text"])
            })
            if len(batch) >= 1000:
                docs_proc.insert_many(batch)
                batch = []
        if batch:
            docs_proc.insert_many(batch)
        print("Processed documents stored")

    def process_queries(self, quries, queries_proc):
        queries_proc.delete_many({})
        batch = []
        for q in quries.find({}):
            batch.append({
                "_id": q["_id"],
                "tokens": self.process(q["text"])
            })
            if len(batch) >= 1000:
                queries_proc.insert_many(batch)
                batch = []
        if batch:
            queries_proc.insert_many(batch)
        print("Processed queries stored")

    def process_data(self):
        self.process_documents(self.database.documents, self.database.documents_processed)
        self.process_queries(self.database.queries, self.database.queries_processed)
        

In [5]:
preprocessor = Preprocessor()
# preprocessor.database.store_documents()
# preprocessor.process_data()
preprocessor.database.summary()

[nltk_data] Downloading package stopwords to /Users/fadi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Documents: 268893
Queries: 417
Qrels: 1376
Documents Processed: 268893
Queries Processed: 417


## TF-IDF

In [6]:
class TFIDFService:
    def __init__(self):
        try:
            self.load_models()
        except FileNotFoundError:
            print("Model files not found. Please run train() first.")
            self.vectorizer = None
            self.tfidf_matrix = None
            self.doc_ids = None

    def load_models(self):
        self.vectorizer = joblib.load("tfidf_vectorizer.joblib")
        self.tfidf_matrix = joblib.load("tfidf_matrix.joblib")
        self.doc_ids = joblib.load("tfidf_doc_ids.joblib")

    def train(self):
        database = DataBase()
        docs_collection = database.documents_processed
        docs = list(docs_collection.find({}))
        docs_df = pd.DataFrame([{
                "doc_id": doc["_id"],
                "tokens_text": " ".join(doc["tokens"])}
            for doc in docs
        ])
        doc_ids = docs_df["doc_id"].tolist()
        corpus = docs_df["tokens_text"].tolist()
        vectorizer = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b\w+\b"
        )
        tfidf_matrix = vectorizer.fit_transform(corpus)
        joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
        joblib.dump(tfidf_matrix, "tfidf_matrix.joblib")
        joblib.dump(doc_ids, "tfidf_doc_ids.joblib")
        print("mission complete!!")
        
    def search(self, query, top_k=10):
        if self.vectorizer is None:
            self.load_models()
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in top_indices:
            results.append(self.doc_ids[idx])
        return results

    def test(self):
        preprocessor = Preprocessor()
        q = preprocessor.process("are zebra loaches safe with shrimp?")
        query = " ".join(q)
        results = self.search(query, top_k=10)
        for r in results:
            print(r)

In [7]:
tfidf_service = TFIDFService()
# tfidf_service.train()
# tfidf_service.test()

Model files not found. Please run train() first.


## Word2Vec

In [8]:
import numpy as np
import joblib

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class Word2VecService:
    def __init__(self):
        try:
            self.load_models()
        except FileNotFoundError:
            self.model = None
            self.doc_vectors = None
            self.doc_ids = None
            print('models not found')
    
    def load_models(self):
        self.model = Word2Vec.load("word2vec_model.model")
        self.doc_vectors = joblib.load(
            "word2vec_doc_vectors.pkl"
        )
        self.doc_ids = joblib.load(
            "word2vec_doc_ids.pkl"
        )

    def document_vector(self, model, tokens):
        vectors = [
            model.wv[word]
            for word in tokens
            if word in model.wv
        ]
        if len(vectors) == 0:
            return np.zeros(model.vector_size)
        return np.mean(vectors, axis=0)

    def train(self):
        database = DataBase()
        docs_collection = database.documents_processed
        docs = list(docs_collection.find({}))
        docs_df = pd.DataFrame([
            {
                "doc_id": doc["_id"],
                "text_clean": " ".join(doc["tokens"])
            }
            for doc in docs
        ])
        doc_ids = docs_df["doc_id"].tolist()
        tokenized_docs = [
            text.split()
            for text in docs_df["text_clean"]
        ]
        model = Word2Vec(
            sentences=tokenized_docs, 
            vector_size=150,   
            window=4,           
            min_count=5,        
            workers=4,          
            sg=0,               
            negative=5,         
            epochs=10
        )
        doc_vectors = np.array([
            self.document_vector(model, tokens)
            for tokens in tokenized_docs
        ])
        model.save("word2vec_model.model")
        joblib.dump(doc_vectors, "word2vec_doc_vectors.pkl")
        joblib.dump(doc_ids, "word2vec_doc_ids.pkl")
        print("mission completed.")

    def query_vector(self, text):
        tokens = text.split()
        vectors = [
            self.model.wv[word]
            for word in tokens
            if word in self.model.wv
        ]
        if len(vectors) == 0:
            return np.zeros(self.model.vector_size)
        return np.mean(vectors, axis=0)

    def search(self, query, top_k=10):
        qvec = self.query_vector(query)
        scores = cosine_similarity(
            [qvec],
            self.doc_vectors
        )[0]
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in top_indices:
            results.append(self.doc_ids[idx])
        return results
    
    def test(self):
        preprocessor = Preprocessor()
        q = preprocessor.process("are zebra loaches safe with shrimp?")
        query = " ".join(q)
        results = self.search(
            query,
            top_k=10
        )
        for r in results:
            print(r)

    def grid_search(self):
        database = DataBase()
        docs_collection = database.documents_processed
        docs = list(docs_collection.find({}))
        docs_df = pd.DataFrame([
            {
                "doc_id": doc["_id"],
                "text_clean": " ".join(doc["tokens"])
            }
            for doc in docs
        ])
        doc_ids = docs_df["doc_id"].tolist()
        tokenized_docs = [
            text.split()
            for text in docs_df["text_clean"]
        ]

        logging.basicConfig(level=logging.WARNING)
        param_grid = {
            'vector_size': [100, 150],
            'window': [4, 5, 6],
            'sg': [0, 1],          # 0 for CBOW, 1 for Skip-gram
            'epochs': [9]
        }

        static_params = {
            'sentences': tokenized_docs,
            'min_count': 5,
            'workers': 4,
            'negative': 5
        }

        # 2. Generate all combinations of hyperparameter values
        keys, values = zip(*param_grid.items())
        experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

        print(f"Total combinations to test: {len(experiments)}\n")

        best_loss = float('inf')
        best_params = None

        # 3. Run the Grid Search Loop
        for i, params in enumerate(experiments, 1):
            # Combine static and grid parameters
            current_config = {**static_params, **params}
            
            print(f"Running experiment {i}/{len(experiments)} with params: {params}")
            
            # We enable compute_loss to track model convergence performance
            model = Word2Vec(
                **current_config,
                compute_loss=True
            )
            
            # Extract the final training loss
            training_loss = model.get_latest_training_loss()
            print(f"-> Training Loss: {training_loss:.2f}")
            
            # Track the best configuration (lower loss is generally better)
            if training_loss < best_loss:
                best_loss = training_loss
                best_params = params


        ### Results

        print("\n" + "="*40)
        print("GRID SEARCH COMPLETE")
        print("="*40)
        print(f"Best Parameters Found: {best_params}")
        print(f"Lowest Training Loss: {best_loss:.2f}")

        ## after running the results were:
        # Best Parameters Found: {'vector_size': 150, 'window': 4, 'sg': 0, 'epochs': 9}
        # Lowest Training Loss: 49414876.00


In [10]:
word2vec_service = Word2VecService()
# word2vec_service.train()

models not found


In [11]:
word2vec_service = Word2VecService()
# word2vec_service.grid_search()
# word2vec_service.test()

models not found


In [12]:
word2vec_service = Word2VecService()
# word2vec_service.test()

models not found


## BM25

In [13]:
import pandas as pd
import joblib
from rank_bm25 import BM25Okapi

In [14]:
class BM25Service:
    def __init__(self):
        try:
            self.load_models()
        except FileNotFoundError:
            print("Model files not found. Please run train() first.")
            self.bm25 = None
            self.doc_ids = None

    def load_models(self):
        self.bm25 = joblib.load("bm25_model.pkl")
        self.doc_ids = joblib.load("bm25_doc_ids.pkl")

    def train(self):
        database = DataBase()
        docs_collection = database.documents_processed
        docs = list(docs_collection.find({}))
        docs_df = pd.DataFrame([
            {
                "doc_id": doc["_id"],
                "text_clean": " ".join(doc["tokens"])
            }
            for doc in docs
        ])
        doc_ids = docs_df["doc_id"].tolist()
        tokenized_corpus = [text.split() for text in docs_df["text_clean"]]
        # tried:
        # bm25 = BM25Okapi(tokenized_corpus, k1=1.4, b=0.64)
        bm25 = BM25Okapi(tokenized_corpus, k1=1.2, b=0.5)
        joblib.dump(bm25, "bm25_model.pkl")
        joblib.dump(doc_ids, "bm25_doc_ids.pkl")
        print("BM25 model saved successfully!")
        print("Documents indexed:", len(doc_ids))

    def search(self, query, top_k=10):
        if self.bm25 is None:
            self.load_models()
        tokens = query.lower().split()
        scores = self.bm25.get_scores(tokens)
        top_indices = scores.argsort()[::-1][:top_k]
        results = []
        for idx in top_indices:
            results.append(self.doc_ids[idx])
        return results

    def test(self):
        preprocessor = Preprocessor()
        q = preprocessor.process("are zebra loaches safe with shrimp?")
        query = " ".join(q)
        results = self.search(
            query,
            top_k=10
        )
        for r in results:
            print(r)

In [15]:
bm25_service = BM25Service()
# bm25_service.train()

Model files not found. Please run train() first.


In [16]:
bm25_service = BM25Service()
# bm25_service.test() 

Model files not found. Please run train() first.


## BERT

In [17]:
import pandas as pd
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer

import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [21]:
class BertService:
    MODEL_NAME = "BAAI/bge-small-en-v1.5"

    def __init__(self):
        try:
            self.load_models()
        except FileNotFoundError:
            print("Model files not found. Please run train() first.")
            self.model = None
            self.doc_vectors = None
            self.doc_ids = None

    def load_models(self):
        self.model = SentenceTransformer(self.MODEL_NAME)
        self.doc_vectors = joblib.load("bert_doc_vectors.pkl")
        self.doc_ids = joblib.load("bert_doc_ids.pkl")

    def train(self):
        database = DataBase()
        docs_collection = database.documents_processed
        docs = list(docs_collection.find({}))
        
        docs_df = pd.DataFrame([
            {
                "doc_id": doc["_id"],
                "text_clean": " ".join(doc["tokens"])
            }
            for doc in docs
        ])
        
        doc_ids = docs_df["doc_id"].tolist()
        corpus = docs_df["text_clean"].tolist()
        
        model = SentenceTransformer(
            self.MODEL_NAME,
            model_kwargs={"torch_dtype": "float16"},
            device="mps"
        )
        model.max_seq_length = 64

        doc_vectors = model.encode(
            corpus,
            batch_size=512,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True
        )
        
        joblib.dump(doc_vectors, "bert_doc_vectors.pkl")
        joblib.dump(doc_ids, "bert_doc_ids.pkl")
        
        print("Training completed.")
        print("Documents indexed:", len(doc_ids))
        print("Embedding dimension:", doc_vectors.shape[1])

    def query_vector(self, text):
        if self.model is None:
            self.load_models()
        return self.model.encode(
            text,
            normalize_embeddings=True,
            max_length=128           
        )

    def search(self, query, top_k=10):
        if self.doc_vectors is None:
            self.load_models()
        
        qvec = self.query_vector(query)
        scores = cosine_similarity(
            [qvec],
            self.doc_vectors
        )[0]
        
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in top_indices:
            results.append(self.doc_ids[idx])
        return results
    
    def test(self):
        results = self.search("zebra loach safe shrimp", top_k=10)
        print(results)
    
    # def test(self):
    #     preprocessor = Preprocessor()
    #     q = preprocessor.process("are zebra loaches safe with shrimp?")
    #     query = " ".join(q)
    #     results = self.search(
    #         query,
    #         top_k=10
    #     )
    #     for r in results:
    #         print(r)

In [22]:
bert_service = BertService()
# bert_service.train()
bert_service.test()

ValueError: SentenceTransformer.encode() has been called with additional keyword arguments that this model does not use: ['max_length']. As per SentenceTransformer.get_model_kwargs(), this model does not accept any additional keyword arguments.

## Hybrid representation

In [ ]:
class HybridParallelBM25Word2Vec:
    def __init__(self):
        self.bm25_search = BM25Service()
        self.word2vec_search = Word2VecService()
        self.doc_ids = self.bm25_search.doc_ids
    
    def search(self, query, top_k=10):
        """
        Retrieve using both BM25 and Word2Vec, combine scores, return top-k
        """
        # Get results and scores from both models
        bm25_tokens = query.lower().split()
        bm25_scores = self.bm25_search.bm25.get_scores(bm25_tokens)
        
        # Word2Vec scores
        w2v_query_vec = self.word2vec_search.query_vector(query)
        w2v_scores = cosine_similarity(
            [w2v_query_vec],
            self.word2vec_search.doc_vectors
        )[0]
        
        # Normalize scores to [0, 1]
        bm25_scores_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-10)
        w2v_scores_norm = (w2v_scores - w2v_scores.min()) / (w2v_scores.max() - w2v_scores.min() + 1e-10)
        
        # Combine with equal weights
        combined_scores = 0.5 * bm25_scores_norm + 0.5 * w2v_scores_norm
        
        # Get top-k results
        top_indices = np.argsort(combined_scores)[::-1][:top_k]
        results = [self.doc_ids[idx] for idx in top_indices]
        
        return results


class HybridSerialBM25BERT:
    def __init__(self, candidate_k=100):
        self.bm25_search = BM25Service()
        self.bert_search = BertService()
        self.candidate_k = candidate_k  # Number of candidates from BM25
        self.doc_ids = self.bm25_search.doc_ids
    
    def search(self, query, top_k=10):
        """
        Retrieve using BM25, then re-rank with BERT
        """
        # Stage 1: BM25 retrieval to get candidates
        bm25_candidates = self.bm25_search.search(query, top_k=self.candidate_k)
        
        # Stage 2: BERT re-ranking on candidates
        query_embedding = self.bert_search.query_vector(query)
        
        # Get embeddings for only the candidate documents
        candidate_indices = [self.doc_ids.index(doc_id) for doc_id in bm25_candidates]
        candidate_embeddings = self.bert_search.doc_vectors[candidate_indices]
        
        # Calculate BERT scores for candidates
        bert_scores = cosine_similarity([query_embedding], candidate_embeddings)[0]
        
        # Sort by BERT scores and return top-k
        sorted_indices = np.argsort(bert_scores)[::-1][:top_k]
        results = [bm25_candidates[idx] for idx in sorted_indices]
        
        return results


In [14]:
# Test Parallel Hybrid (BM25 + Word2Vec)
hybrid_parallel = HybridParallelBM25Word2Vec()
test_query = "zebra loach safe shrimp"
results_parallel = hybrid_parallel.search(test_query, top_k=10)
print("Parallel Hybrid (BM25 + Word2Vec):")
for i, r in enumerate(results_parallel, 1):
    print(f"  {i}. {r}")

print("\n" + "="*50 + "\n")

# Test Serial Hybrid (BM25 → BERT)
hybrid_serial = HybridSerialBM25BERT(candidate_k=50)
results_serial = hybrid_serial.search(test_query, top_k=10)
print("Serial Hybrid (BM25 → BERT):")
for i, r in enumerate(results_serial, 1):
    print(f"  {i}. {r}")


Parallel Hybrid (BM25 + Word2Vec):
  1. 7323
  2. 3811
  3. 6305
  4. 619
  5. 4573
  6. 8586
  7. 7794
  8. 70166
  9. 451
  10. 7103


Serial Hybrid (BM25 → BERT):
  1. 7323
  2. 23561
  3. 4162
  4. 5159
  5. 3811
  6. 4285
  7. 619
  8. 6548
  9. 2763
  10. 2053


## Evaluation

In [ ]:
import math
from collections import defaultdict

In [ ]:
class EvaluationService:
    def __init__(self):
        self.db = DataBase()
        self.qrels = self.db.qrels
        self.queries = self.db.queries
        self.documents = self.db.documents
        self.documents_processed = self.db.documents_processed

        self.tfidf_search = TFIDFService()
        self.word2vec_search = Word2VecService()
        self.bm25_search = BM25Service()
        self.bert_search = BertService()
        self.hybrid_parallel = HybridParallelBM25Word2Vec()
        self.hybrid_serial = HybridSerialBM25BERT(candidate_k=50)


    def compute_query_metrics(self, ranked_list: list, qrel_list: list, top_k_p: int = 10) -> dict:
        """
        Computes Precision@K, Recall, Average Precision (for MAP), and nDCG 
        for a single query sequence.
        """
        # Total actual relevant documents in the ground truth for this query (relevance > 0)
        total_relevant = sum(1 for rel in qrel_list)
        
        if total_relevant == 0:
            return None # Skip queries that have no true positive judgments

        # 1. Precision @ K (e.g., P@10)
        top_k_docs = ranked_list[:top_k_p]
        rel_in_top_k = sum(1 for doc_id in top_k_docs if doc_id in qrel_list)
        precision_at_k = rel_in_top_k / top_k_p

        # 2. Recall (evaluated across the total retrieved ranked list)
        rel_retrieved = sum(1 for doc_id in ranked_list if doc_id in qrel_list)
        recall = rel_retrieved / total_relevant

        # 3. Average Precision (AP) -> used to calculate Mean Average Precision (MAP)
        ap_sum = 0.0
        num_rel_found = 0
        for rank, doc_id in enumerate(ranked_list, start=1):
            if doc_id in qrel_list:
                num_rel_found += 1
                precision_at_rank = num_rel_found / rank
                ap_sum += precision_at_rank
        ap = ap_sum / total_relevant

        # 4. Normalized Discounted Cumulative Gain (nDCG)
        # Calculate Discounted Cumulative Gain (DCG)
        dcg = 0.0
        for rank, doc_id in enumerate(ranked_list, start=1):
            rel = 1 if doc_id in qrel_list else 0
            dcg += (2**rel - 1) / math.log2(rank + 1)

        # Calculate Ideal DCG (IDCG) based on the best possible sorting of true judgments
        ideal_rels = sorted([1 if doc_id in qrel_list else 0 for doc_id in ranked_list], reverse=True)
        idcg = 0.0
        for rank, rel in enumerate(ideal_rels[:len(ranked_list)], start=1):
            idcg += (2**rel - 1) / math.log2(rank + 1)

        ndcg = dcg / idcg if idcg > 0 else 0.0

        return {
            "p_at_k": precision_at_k,
            "recall": recall,
            "ap": ap,
            "ndcg": ndcg
        }

    def run(self):
        print("Loading qrels from MongoDB...")
        qrels_lookup = defaultdict(list)
        for qrel in self.qrels.find(
                {},
                {"_id": 0, "query_id": 1, "doc_id": 1, "relevance": 1}
            ):
            if qrel.get("relevance", 0) > 0:
                qrels_lookup[qrel["query_id"]].append(str(qrel["doc_id"]))

        print("Loading preprocessed queries...")
        queries = list(
            self.queries.find(
                {},
                {"_id": 0, "query_id": 1, "text_clean": 1}
            )
        )
        print(f"Loaded {len(queries)} queries and {len(qrels_lookup)} qrel mappings.\n")
        models_to_evaluate = {
            "TF-IDF": self.tfidf_search.search,
            "Word2Vec": self.word2vec_search.search,
            "BM25": self.bm25_search.search,
            "BERT": self.bert_search.search,
            "Parallel (BM25+W2V)": self.hybrid_parallel.search,
            "Serial (BM25→BERT)": self.hybrid_serial.search
        }
        final_results = {}
        for model_name, retrieve_fn in models_to_evaluate.items():
            print(f"Running evaluation for model: {model_name}...")

            total_p_at_10 = 0.0
            total_recall = 0.0
            total_ap = 0.0
            total_ndcg = 0.0
            evaluated_queries_count = 0

            for q in queries:
                q_id = q["query_id"]
                q_text = q["text_clean"]

                if q_id not in qrels_lookup:
                    continue

                ranked_results = retrieve_fn(
                    query=q_text,
                    top_k=100
                )

                if not ranked_results:
                    ranked_results = []

                metrics = self.compute_query_metrics(
                    ranked_results,
                    qrels_lookup[q_id],
                    top_k_p=10
                )

                if metrics:
                    total_p_at_10 += metrics["p_at_k"]
                    total_recall += metrics["recall"]
                    total_ap += metrics["ap"]
                    total_ndcg += metrics["ndcg"]
                    evaluated_queries_count += 1

            if evaluated_queries_count > 0:
                final_results[model_name] = {
                    "MAP": total_ap / evaluated_queries_count,
                    "Recall": total_recall / evaluated_queries_count,
                    "Precision@10": total_p_at_10 / evaluated_queries_count,
                    "nDCG": total_ndcg / evaluated_queries_count
                }
            else:
                final_results[model_name] = {
                    "MAP": 0.0,
                    "Recall": 0.0,
                    "Precision@10": 0.0,
                    "nDCG": 0.0
                }

        print("\n======================= EVALUATION REPORT =======================")
        print(f"{'Model':<20} | {'MAP':<10} | {'Recall':<10} | {'Precision@10':<12} | {'nDCG':<10}")
        print("-" * 75)

        for model_name, metrics in final_results.items():
            print(
                f"{model_name:<20} | "
                f"{metrics['MAP']:<10.4f} | "
                f"{metrics['Recall']:<10.4f} | "
                f"{metrics['Precision@10']:<12.4f} | "
                f"{metrics['nDCG']:<10.4f}"
            )

        print("===================================================================")

Loading qrels from database...
Loading preprocessed queries...
Loaded 417 queries and 417 qrel mappings.

Running evaluation for model: TF-IDF...
Running evaluation for model: Word2Vec...
Running evaluation for model: BM25...
Running evaluation for model: BERT...
Running evaluation for model: Parallel (BM25+W2V)...
Running evaluation for model: Serial (BM25→BERT)...

======================= EVALUATION REPORT =======================
Model                | MAP        | Recall     | Precision@10 | nDCG      
---------------------------------------------------------------------------
TF-IDF               | 0.1904     | 0.5341     | 0.0813       | 0.3772    
Word2Vec             | 0.1568     | 0.5385     | 0.0741       | 0.3419    
BM25                 | 0.2822     | 0.6433     | 0.1086       | 0.4879    
BERT                 | 0.1667     | 0.5574     | 0.0748       | 0.3587    
Parallel (BM25+W2V)  | 0.2765     | 0.6549     | 0.1108       | 0.4820    
Serial (BM25→BERT)   | 0.2162     | 0.